<a href="https://colab.research.google.com/github/LucasGABernardo/Lista3/blob/main/Lista3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, ConfusionMatrixDisplay

In [ ]:
df = pd.read_csv('dataset_emprestimo_aprovacao.csv')

# Exibir informações de tipo e nulos
print("--- Visão Geral da Estrutura das Colunas ---")
print(df.info())

# Estatísticas básicas descritivas
print("\n--- Estatísticas Descritivas Gerais ---")
print(df.describe())

# Verificar se as classes estão balanceadas ou desbalanceadas
print("\n--- Distribuição da Variável Alvo (emprestimo_aprovado) ---")
print(df['emprestimo_aprovado'].value_counts())
print("Proporção: Aprovados (1) = 79% | Não Aprovados (0) = 21%")

# Visualização de Dispersão dos Clientes
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x='score_credito',
    y='renda_mensal',
    hue=df['emprestimo_aprovado'].map({0: 'Não Aprovado', 1: 'Aprovado'}),
    palette={ 'Aprovado': '#2ecc71', 'Não Aprovado': '#e74c3c' },
    s=100,
    alpha=0.8
)
plt.title('Distribuição de Clientes: Score de Crédito vs Renda Mensal', fontsize=14, fontweight='bold')
plt.xlabel('Score de Crédito', fontsize=12)
plt.ylabel('Renda Mensal (R$)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Separar features (entradas) e target (alvo)
X = df.drop(columns=['emprestimo_aprovado'])
y = df['emprestimo_aprovado']

# 2. Divisão em Treino (80%) e Teste (20%) com amostragem estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Formato da base de Treino (X): {X_train.shape}")
print(f"Formato da base de Teste (X): {X_test.shape}")

In [ ]:
# Instanciar o normalizador
scaler = StandardScaler()

# Ajustar e transformar com os dados de treino; apenas transformar os dados de teste
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Exemplo de dados originais (Primeira linha de Treino):\n", X_train.iloc[0].values)
print("\nExemplo de dados após normalização (Z-score correspondente):\n", X_train_scaled[0])

In [ ]:
# Criar e treinar o classificador KNN
modelo_knn = KNeighborsClassifier(n_neighbors=5)
modelo_knn.fit(X_train_scaled, y_train)

# Executar previsões na base de teste
y_pred = modelo_knn.predict(X_test_scaled)

# 1. Calcular acurácia global
acuracia = accuracy_score(y_test, y_pred)
print(f"Acurácia Global do Modelo: {acuracia:.2%}\n")

# 2. Exibir relatório detalhado de métricas por classe
print("--- Relatório Completo de Classificação ---")
print(classification_report(y_test, y_pred, target_names=['Não Aprovado', 'Aprovado']))

# 3. Plotar a matriz de confusão de forma legível
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Não Aprovado', 'Aprovado'])

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap='Blues', values_format='d', ax=ax)
plt.title('Matriz de Confusão - Classificador KNN', fontsize=12, fontweight='bold')
plt.grid(False)
plt.show()

In [ ]:
# Definir um cliente de teste
# Renda Mensal: R$ 5.000 | Score de Crédito: 450 | Dívidas Ativas: 4
novo_cliente = pd.DataFrame([{
    'renda_mensal': 5000.00,
    'score_credito': 450,
    'dividas_ativas': 4
}])

# ATENÇÃO: O dado deve passar obrigatoriamente pelo mesmo scaler antes de ir para o modelo!
novo_cliente_scaled = scaler.transform(novo_cliente)

# Predição
predicao = modelo_knn.predict(novo_cliente_scaled)
probabilidade = modelo_knn.predict_proba(novo_cliente_scaled)

resultado = "APROVADO" if predicao[0] == 1 else "REPROVADO"
print(f"Diagnóstico para o Novo Cliente: {resultado}")
print(f"Confiança do KNN (Votação dos Vizinhos): {probabilidade[0][predicao[0]]*100}% de certeza.")